### **1. Entender a Interface RiskFilter**
Seu filtro de risco (como `liquidity_filter`) segue este padrão:
- **Entrada**: `weights` (DataFrame com posições) + `ibov` (Series de bench)
- **Saída**: `weights` modificado (com posições reduzidas para ativos de risco alto)

```python
# Exemplo: liquidity_filter
def liquidity_filter(volume, window=20, min_adv=500_000) -> RiskFilter:
    # ... calcula ADV ...
    def _filter(weights: pd.DataFrame, ibov: pd.Series) -> pd.DataFrame:
        # reduz pesos onde volume < min_adv
        return weights.where(mask, 0.0)
    return _filter
```

### **2. Como usar netConnectivity para reduzir posições**

O `netConnectivity` calcula 3 sinais importantes:
- **`norm_eigen_vector`** → Ranking de conectividade de cada ativo (0 a 1)
  - Valor **alto** = ativo muito conectado = **risco sistêmico**
  - Valor **baixo** = ativo pouco conectado = **mais seguro**
- **`max_eigen_value`** → Magnitude da fragilidade sistêmica geral
  - Se cresce rapidamente = aumento de cluster/concentração
- **`density`** → Quantidade de correlações acima do threshold

### **3. Implementação Prática**

Crie um novo filtro em portfolio que:

1. **Calcula `net_analysis()` com uma janela móvel** (ex: últimos 252 dias)
2. **Reduz posições baseado no `norm_eigen_vector`**:
   ```
   novo_peso = peso_original × (1 - conectividade_normalizada × força_do_filtro)
   ```
   Exemplo: Se ativo tem eigenvector=0.9 e força=0.5, reduz para 55% do peso

3. **Opcionalmente, ativa modo "risco alto"** quando `max_eigen_value` ou `density` excedem limiar:
   - Em períodos de conexão alta → reduz exposição geral (ex: 50%)
   - Útil para combinar com regimes de alta volatilidade

### **4. Integração no seu main.py**

Seria assim:

```python
# Importar seu novo filtro
from portfolio import liquidity_filter, net_connectivity_filter  # (novo)

# No main(), substituir:
risk_filters = [
    liquidity_filter(volume, window=20, min_adv=500_000),
    net_connectivity_filter(prices, threshold=0.7, connectivity_weight=0.5)  # novo
]

result = run_pipeline(prices, ibov, regime_signal, strategy, risk_filters)
```

### **5. Parâmetros-chave a testar**

| Parâmetro | Significado | Range |
|-----------|-------------|-------|
| `threshold` | Correlação mínima para contar como "conectado" | 0.5-0.8 |
| `connectivity_weight` | Força do filtro (quanto reduz posições) | 0-1.0 |
| `eigen_threshold` | Ativa filtro só se fragilidade > valor | Optional |

### **6. Estratégias de aplicação**

**Opção A - Simples (reduzir sempre)**
- Reduz posições em ativos muito conectados proporcionalmente

**Opção B - Regime-aware (combinar com Markov)**
- Use netConnectivity como **filtro extra apenas em HIGH_VOL regime**
- Quando volatilidade é alta + conectividade é alta = máximo risco

**Opção C - Dinâmica (janela móvel)**
- Recalcule `net_analysis()` mensalmente ou trimestralmente
- Adapta-se a mudanças na estructura de correlação